In [ ]:
import pandas as pd
import pytz
import datetime as dt
from sklearn.linear_model import LinearRegression
import joblib
from sklearn.preprocessing import StandardScaler

In [ ]:
def preprocess_QuantAQ(file_path, start_date, end_date):
    sensor = pd.read_csv(file_path, parse_dates=['timestamp'])
    sensor = sensor[['timestamp','rh','temp','pm10']].dropna(subset=['pm10'])
    sensor.rename(columns={'timestamp':'time'}, inplace=True)
    
    sensor['time'] = pd.to_datetime(sensor['time'])
    est = pytz.timezone('US/Eastern')
    sensor['time'] = sensor['time'].dt.tz_convert(est)
    sensor['time'] = sensor['time'].dt.tz_localize(None)
    sensor['day'] = sensor['time'].dt.date
    sensor['dayhour'] = sensor['time'].dt.strftime('%Y-%m-%d %H')
    
    sensor = sensor[(sensor['time'] >= start_date) & (sensor['time'] <= end_date)]


    sensor = sensor[::-1].reset_index(drop=True)
    # train only on last 3/4 of data
    # split_index = int(len(sensor) * 0.25)
    # sensor = sensor.iloc[split_index:]


    return sensor


In [ ]:
def create_enhanced_features(df, hourly=True):
    features = df.copy()

    if hourly:
        # Original features
        features['pm10'] = df['shourlyPM10']
        features['rh'] = df['shourlyRH']
        features['temp'] = df['shourlyTemp']
        
        # Interaction terms - these capture how RH/temp affect bias differently at different PM levels
        features['pm10_x_rh'] = df['shourlyPM10'] * df['shourlyRH']
        features['pm10_x_temp'] = df['shourlyPM10'] * df['shourlyTemp']
        features['rh_x_temp'] = df['shourlyRH'] * df['shourlyTemp']
        
        # Polynomial terms - capture non-linear effects
        features['pm10_sq'] = df['shourlyPM10'] ** 2
        features['rh_sq'] = df['shourlyRH'] ** 2
        features['temp_sq'] = df['shourlyTemp'] ** 2
    else:
        # Original features
        features['pm10'] = df['sdailyPM10']
        features['rh'] = df['sdailyRH']
        features['temp'] = df['sdailyTemp']
        
        # Interaction terms - these capture how RH/temp affect bias differently at different PM levels
        features['pm10_x_rh'] = df['sdailyPM10'] * df['sdailyRH']
        features['pm10_x_temp'] = df['sdailyPM10'] * df['sdailyTemp']
        features['rh_x_temp'] = df['sdailyRH'] * df['sdailyTemp']
        
        # Polynomial terms - capture non-linear effects
        features['pm10_sq'] = df['sdailyPM10'] ** 2
        features['rh_sq'] = df['sdailyRH'] ** 2
        features['temp_sq'] = df['sdailyTemp'] ** 2
    
    return features

In [ ]:
# def create_enhanced_features2(df, hourly=True):
#     """
#     Creates enhanced features for PM10 bias correction.
    
#     PM10-specific considerations:
#     - Includes coarse particles (2.5-10 μm) from dust, pollen, mechanical processes
#     - More influenced by resuspension and physical disturbances
#     - PM10/PM2.5 ratio is highly informative about source types
#     - Coarse fraction (PM10 - PM2.5) helps identify non-combustion sources
    
#     Feature groups:
#     1. Core measurements
#     2. Temporal patterns (daily/weekly cycles)
#     3. Multi-pollutant features (source indicators)
#     4. Meteorological derived (particle behavior)
#     5. PM size distribution (PM10-specific insight)
#     6. Interaction terms
#     7. Polynomial terms
#     8. Trend features
#     """
#     features = df.copy()
    
#     if hourly:
#         # ===== 1. CORE MEASUREMENTS =====
#         features['pm10'] = df['shourlyPM10']
#         features['rh'] = df['shourlyRH']
#         features['temp'] = df['shourlyTemp']
        
#         # ===== 2. TEMPORAL PATTERNS =====
#         # Why: PM10 has daily patterns (morning dust resuspension, traffic, evening settling)
#         features['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
#         features['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
#         features['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
#         features['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
#         features['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        
#         # ===== 3. MULTI-POLLUTANT FEATURES =====
#         # Why: Help distinguish combustion PM (high PM2.5/CO/NO2) from dust PM (high coarse fraction)
#         features['pm1'] = df['shourlyPM1']
#         features['pm25'] = df['shourlyPM25']
#         features['co'] = df['shourlyCO']
#         features['no2'] = df['shourlyNO2']
#         features['o3'] = df['shourlyO3']
        
#         # ===== 4. METEOROLOGICAL DERIVED =====
#         # Why: RH affects both fine and coarse particles, but differently
        
#         # Dew Point
#         a, b = 17.27, 237.7
#         alpha = ((a * features['temp']) / (b + features['temp'])) + np.log(features['rh']/100.0 + 0.001)
#         features['dew_point'] = (b * alpha) / (a - alpha)
        
#         # Vapor Pressure Deficit
#         features['vpd'] = 0.611 * np.exp((17.27 * features['temp']) / (features['temp'] + 237.3)) * (1 - features['rh']/100)
        
#         # Temperature - Dew Point spread
#         features['temp_dewpoint_diff'] = features['temp'] - features['dew_point']
        
#         # ===== 5. PM SIZE DISTRIBUTION (PM10-SPECIFIC - MOST IMPORTANT!) =====
#         # Why: These are CRUCIAL for PM10 - they tell us about particle sources
        
#         # Coarse Fraction (PM10 - PM2.5): mechanical sources like dust, pollen
#         features['coarse_fraction'] = df['shourlyPM10'] - df['shourlyPM25']
#         features['coarse_fraction'] = features['coarse_fraction'].clip(lower=0)  # Can't be negative
        
#         # Fine Fraction for reference
#         features['fine_fraction'] = df['shourlyPM25']
        
#         # Coarse-to-total ratio (HIGH = dust/mechanical, LOW = combustion-dominated)
#         features['coarse_ratio'] = features['coarse_fraction'] / (df['shourlyPM10'] + 0.001)
        
#         # PM10/PM2.5 ratio (similar insight, different scale)
#         features['pm10_pm25_ratio'] = df['shourlyPM10'] / (df['shourlyPM25'] + 0.001)
        
#         # PM2.5/PM1 ratio (fine mode distribution)
#         features['pm25_pm1_ratio'] = df['shourlyPM25'] / (df['shourlyPM1'] + 0.001)
        
#         # Super-coarse indicator (when PM10/PM2.5 > 2, likely dust event)
#         features['is_dust_event'] = (features['pm10_pm25_ratio'] > 2.0).astype(int)
        
#         # ===== 6. INTERACTION TERMS =====
#         # Why: Bias may depend on combinations (e.g., high PM + high RH)
        
#         # Core interactions
#         features['pm10_x_rh'] = df['shourlyPM10'] * df['shourlyRH']
#         features['pm10_x_temp'] = df['shourlyPM10'] * df['shourlyTemp']
#         features['rh_x_temp'] = df['shourlyRH'] * df['shourlyTemp']
        
#         # PM10-specific interactions
#         features['coarse_x_rh'] = features['coarse_fraction'] * df['shourlyRH']  # Coarse particles + humidity
#         features['coarse_x_temp'] = features['coarse_fraction'] * df['shourlyTemp']
#         features['pm10_x_no2'] = df['shourlyPM10'] * df['shourlyNO2']  # Total PM + traffic marker
        
#         # ===== 7. POLYNOMIAL TERMS =====
#         # Why: Non-linear sensor response, especially at high concentrations
#         features['pm10_sq'] = df['shourlyPM10'] ** 2
#         features['rh_sq'] = df['shourlyRH'] ** 2
#         features['temp_sq'] = df['shourlyTemp'] ** 2
#         features['coarse_sq'] = features['coarse_fraction'] ** 2
        
#         # ===== 8. TREND/ROLLING FEATURES =====
#         # Why: Capture buildup or clearance of particles
#         features['pm10_roll_3h'] = df['shourlyPM10_roll_3h']
#         features['pm10_roll_6h'] = df['shourlyPM10_roll_6h']
#         features['pm10_std_3h'] = df['shourlyPM10_std_3h']  # Variability
        
#         # Coarse fraction trends (unique to PM10!)
#         features['coarse_roll_3h'] = df['shourlyCoarse_roll_3h']
        
#     else:  # DAILY MODEL
#         # ===== 1. CORE MEASUREMENTS =====
#         features['pm10'] = df['sdailyPM10']
#         features['rh'] = df['sdailyRH']
#         features['temp'] = df['sdailyTemp']
        
#         # ===== 2. TEMPORAL PATTERNS =====
#         features['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
#         features['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
#         features['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
        
#         # ===== 3. MULTI-POLLUTANT =====
#         features['pm1'] = df['sdailyPM1']
#         features['pm25'] = df['sdailyPM25']
#         features['co'] = df['sdailyCO']
#         features['no2'] = df['sdailyNO2']
#         features['o3'] = df['sdailyO3']
        
#         # ===== 4. METEOROLOGICAL DERIVED =====
#         a, b = 17.27, 237.7
#         alpha = ((a * features['temp']) / (b + features['temp'])) + np.log(features['rh']/100.0 + 0.001)
#         features['dew_point'] = (b * alpha) / (a - alpha)
#         features['vpd'] = 0.611 * np.exp((17.27 * features['temp']) / (features['temp'] + 237.3)) * (1 - features['rh']/100)
#         features['temp_dewpoint_diff'] = features['temp'] - features['dew_point']
        
#         # ===== 5. PM SIZE DISTRIBUTION =====
#         features['coarse_fraction'] = df['sdailyPM10'] - df['sdailyPM25']
#         features['coarse_fraction'] = features['coarse_fraction'].clip(lower=0)
#         features['fine_fraction'] = df['sdailyPM25']
#         features['coarse_ratio'] = features['coarse_fraction'] / (df['sdailyPM10'] + 0.001)
#         features['pm10_pm25_ratio'] = df['sdailyPM10'] / (df['sdailyPM25'] + 0.001)
#         features['pm25_pm1_ratio'] = df['sdailyPM25'] / (df['sdailyPM1'] + 0.001)
#         features['is_dust_event'] = (features['pm10_pm25_ratio'] > 2.0).astype(int)
        
#         # ===== 6. INTERACTIONS =====
#         features['pm10_x_rh'] = df['sdailyPM10'] * df['sdailyRH']
#         features['pm10_x_temp'] = df['sdailyPM10'] * df['sdailyTemp']
#         features['rh_x_temp'] = df['sdailyRH'] * df['sdailyTemp']
#         features['coarse_x_rh'] = features['coarse_fraction'] * df['sdailyRH']
#         features['coarse_x_temp'] = features['coarse_fraction'] * df['sdailyTemp']
#         features['pm10_x_no2'] = df['sdailyPM10'] * df['sdailyNO2']
        
#         # ===== 7. POLYNOMIALS =====
#         features['pm10_sq'] = df['sdailyPM10'] ** 2
#         features['rh_sq'] = df['sdailyRH'] ** 2
#         features['temp_sq'] = df['sdailyTemp'] ** 2
#         features['coarse_sq'] = features['coarse_fraction'] ** 2
        
#         # ===== 8. TRENDS =====
#         features['pm10_roll_3h'] = df['sdailyPM10_roll_3h']
#         features['pm10_roll_6h'] = df['sdailyPM10_roll_6h']
#         features['pm10_std_3h'] = df['sdailyPM10_std_3h']
#         features['coarse_roll_3h'] = df['sdailyCoarse_roll_3h']
    
#     return features

In [ ]:
# def preprocess_QuantAQ2(file_path, start_date, end_date):
#     sensor = pd.read_csv(file_path, parse_dates=['timestamp'])
#     # Include all PM sizes and pollutants
#     sensor = sensor[['timestamp','rh','temp','pm1','pm25','pm10','co','no2','o3']].dropna(subset=['pm10'])
#     sensor.rename(columns={'timestamp':'time'}, inplace=True)
    
#     sensor['time'] = pd.to_datetime(sensor['time'])
#     est = pytz.timezone('US/Eastern')
#     sensor['time'] = sensor['time'].dt.tz_convert(est)
#     sensor['time'] = sensor['time'].dt.tz_localize(None)
    
#     # Sort for rolling features
#     sensor = sensor.sort_values('time').reset_index(drop=True)
    
#     # Calculate coarse fraction early
#     sensor['coarse_fraction'] = (sensor['pm10'] - sensor['pm25']).clip(lower=0)
    
#     # Rolling features for PM10
#     sensor['pm10_roll_3h'] = sensor['pm10'].rolling(window=3, min_periods=1).mean()
#     sensor['pm10_roll_6h'] = sensor['pm10'].rolling(window=6, min_periods=1).mean()
#     sensor['pm10_std_3h'] = sensor['pm10'].rolling(window=3, min_periods=1).std().fillna(0)
    
#     # Rolling features for coarse fraction (PM10-specific insight!)
#     sensor['coarse_roll_3h'] = sensor['coarse_fraction'].rolling(window=3, min_periods=1).mean()
    
#     sensor['day'] = sensor['time'].dt.date
#     sensor['dayhour'] = sensor['time'].dt.strftime('%Y-%m-%d %H')
#     sensor['hour'] = sensor['time'].dt.hour
#     sensor['day_of_week'] = sensor['time'].dt.dayofweek
    
#     sensor = sensor[(sensor['time'] >= start_date) & (sensor['time'] <= end_date)]
#     sensor = sensor[::-1].reset_index(drop=True)
    
#     return sensor

In [ ]:
# def train_hourly_model2(sensor_collocation_data, h_regular):
#     temp = sensor_collocation_data.groupby('dayhour').agg(
#         shourlyPM10=('pm10', lambda x: x.mean(skipna=True)),
#         shourlyPM25=('pm25', lambda x: x.mean(skipna=True)),
#         shourlyPM1=('pm1', lambda x: x.mean(skipna=True)),
#         shourlyRH=('rh', lambda x: x.mean(skipna=True)),
#         shourlyTemp=('temp', lambda x: x.mean(skipna=True)),
#         shourlyCO=('co', lambda x: x.mean(skipna=True)),
#         shourlyNO2=('no2', lambda x: x.mean(skipna=True)),
#         shourlyO3=('o3', lambda x: x.mean(skipna=True)),
#         shourlyPM10_roll_3h=('pm10_roll_3h', lambda x: x.mean(skipna=True)),
#         shourlyPM10_roll_6h=('pm10_roll_6h', lambda x: x.mean(skipna=True)),
#         shourlyPM10_std_3h=('pm10_std_3h', lambda x: x.mean(skipna=True)),
#         shourlyCoarse_roll_3h=('coarse_roll_3h', lambda x: x.mean(skipna=True)),
#         hour=('hour', 'first'),
#         day_of_week=('day_of_week', 'first')
#     ).reset_index()

#     merged = pd.merge(temp, h_regular, on='dayhour').dropna()
#     X = create_enhanced_features(merged)
    
#     # PM10-optimized feature list
#     feature_cols = [
#         # Core
#         'pm10', 'rh', 'temp',
#         # Multi-pollutant
#         'pm25', 'pm1', 'co', 'no2', 'o3',
#         # Temporal
#         'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'is_weekend',
#         # Size distribution (MOST IMPORTANT FOR PM10!)
#         'coarse_fraction', 'coarse_ratio', 'pm10_pm25_ratio', 'is_dust_event',
#         # Meteorological
#         'dew_point', 'vpd', 'temp_dewpoint_diff',
#         # Trends
#         'pm10_roll_3h', 'pm10_roll_6h', 'pm10_std_3h', 'coarse_roll_3h',
#         # Interactions
#         'pm10_x_rh', 'pm10_x_temp', 'rh_x_temp', 'coarse_x_rh', 'pm10_x_no2',
#         # Polynomials
#         'pm10_sq', 'rh_sq', 'temp_sq', 'coarse_sq'
#     ]
    
#     target = merged['rhourlyPM10']

#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X[feature_cols])
    
#     # Use Gradient Boosting for better fit
#     from sklearn.ensemble import GradientBoostingRegressor
#     model = GradientBoostingRegressor(
#         n_estimators=100,
#         learning_rate=0.1,
#         max_depth=5,
#         random_state=42
#     )
#     model.fit(X_scaled, target)

#     return {
#         'model': model,
#         'scaler': scaler,
#         'feature_cols': feature_cols
#     }

# def train_daily_model(sensor_collocation_data, d_regular):
#     temp = sensor_collocation_data.groupby('day').agg(
#         sdailyPM10=('pm10', lambda x: x.mean(skipna=True)),
#         sdailyPM25=('pm25', lambda x: x.mean(skipna=True)),
#         sdailyPM1=('pm1', lambda x: x.mean(skipna=True)),
#         sdailyRH=('rh', lambda x: x.mean(skipna=True)),
#         sdailyTemp=('temp', lambda x: x.mean(skipna=True)),
#         sdailyCO=('co', lambda x: x.mean(skipna=True)),
#         sdailyNO2=('no2', lambda x: x.mean(skipna=True)),
#         sdailyO3=('o3', lambda x: x.mean(skipna=True)),
#         sdailyPM10_roll_3h=('pm10_roll_3h', lambda x: x.mean(skipna=True)),
#         sdailyPM10_roll_6h=('pm10_roll_6h', lambda x: x.mean(skipna=True)),
#         sdailyPM10_std_3h=('pm10_std_3h', lambda x: x.mean(skipna=True)),
#         sdailyCoarse_roll_3h=('coarse_roll_3h', lambda x: x.mean(skipna=True)),
#         day_of_week=('day_of_week', 'first')
#     ).reset_index()

#     merged = pd.merge(temp, d_regular, on='day').dropna().sort_values('day')
#     X = create_enhanced_features(merged, hourly=False)
    
#     feature_cols = [
#         'pm10', 'rh', 'temp',
#         'pm25', 'pm1', 'co', 'no2', 'o3',
#         'day_sin', 'day_cos', 'is_weekend',
#         'coarse_fraction', 'coarse_ratio', 'pm10_pm25_ratio', 'is_dust_event',
#         'dew_point', 'vpd', 'temp_dewpoint_diff',
#         'pm10_roll_3h', 'pm10_roll_6h', 'pm10_std_3h', 'coarse_roll_3h',
#         'pm10_x_rh', 'pm10_x_temp', 'rh_x_temp', 'coarse_x_rh', 'pm10_x_no2',
#         'pm10_sq', 'rh_sq', 'temp_sq', 'coarse_sq'
#     ]
    
#     target = merged['rdailyPM10']

#     scaler = StandardScaler()
#     X_scaled = scaler.fit_transform(X[feature_cols])
    
#     from sklearn.ensemble import GradientBoostingRegressor
#     model = GradientBoostingRegressor(
#         n_estimators=100,
#         learning_rate=0.1,
#         max_depth=5,
#         random_state=42
#     )
#     model.fit(X_scaled, target)

#     return {
#         'model': model,
#         'scaler': scaler,
#         'feature_cols': feature_cols
#     }

In [ ]:
# returns a model that predicts daily pm10 from hourly sensor data
def train_hourly_model(sensor_collocation_data, h_regular):
    temp = sensor_collocation_data.groupby('dayhour').agg(
        shourlyPM10=('pm10', lambda x: x.mean(skipna=True)),
        shourlyRH=('rh', lambda x: x.mean(skipna=True)),
        shourlyTemp=('temp', lambda x: x.mean(skipna=True))
    ).reset_index()

    # inner join (default) - only keep rows that are in both dataframes
    merged = pd.merge(temp, h_regular, on='dayhour').dropna()
    X = create_enhanced_features(merged)
    feature_cols = ['pm10', 'rh', 'temp', 'pm10_x_rh', 'pm10_x_temp', 
                       'rh_x_temp', 'pm10_sq', 'rh_sq', 'temp_sq']
    target = merged['rhourlyPM10']

    scalar = StandardScaler()
    X_scaled = scalar.fit_transform(X[feature_cols])
    model = LinearRegression().fit(X_scaled, target)

    return {
        'model': model,
        'scaler': scalar,
        'feature_cols': feature_cols
    }

def train_daily_model(sensor_collocation_data, d_regular):
    temp = sensor_collocation_data.groupby('day').agg(
        sdailyPM10=('pm10', lambda x: x.mean(skipna=True)),
        sdailyRH=('rh', lambda x: x.mean(skipna=True)),
        sdailyTemp=('temp', lambda x: x.mean(skipna=True))
    ).reset_index()

    # inner join (default) - only keep rows that are in both dataframes
    merged = pd.merge(temp, d_regular, on='day').dropna().sort_values('day')
    X = create_enhanced_features(merged, hourly=False)
    feature_cols = ['pm10', 'rh', 'temp', 'pm10_x_rh', 'pm10_x_temp', 
                       'rh_x_temp', 'pm10_sq', 'rh_sq', 'temp_sq']
    target = merged['rdailyPM10']

    scalar = StandardScaler()
    X_scaled = scalar.fit_transform(X[feature_cols])
    model = LinearRegression().fit(X_scaled, target)

    return {
        'model': model,
        'scaler': scalar,
        'feature_cols': feature_cols
    }
    



In [ ]:
quantAQ00589 = preprocess_QuantAQ("../ShortenedData/MOD-00589-RAW.csv", '2025-06-01', '2025-07-14')
quantAQ00590 = preprocess_QuantAQ("../ShortenedData/MOD-00590-RAW.csv", '2025-06-01', '2025-07-14')
quantAQ00591 = preprocess_QuantAQ("../ShortenedData/MOD-00591-RAW.csv", '2025-06-01', '2025-07-14')
quantAQ00592 = preprocess_QuantAQ("../ShortenedData/MOD-00592-RAW.csv", '2025-06-01', '2025-07-14')
quantAQ00593 = preprocess_QuantAQ("../ShortenedData/MOD-00593-RAW.csv", '2025-06-01', '2025-07-14')
quantAQ00595 = preprocess_QuantAQ("../ShortenedData/MOD-00595-RAW.csv", '2025-06-01', '2025-07-14')

In [ ]:
# load and clean GAPA data
gapa = pd.read_csv('PM10&PM2.5.csv', header=2)
gapa.columns = ['Date', 'PMHR_2', 'PMHR', 'PMHR10', '24H_PMHR10', 'PMHRC', '24H_PMHR']
gapa = gapa[['Date', 'PMHR10']]

gapa['time'] = pd.to_datetime(gapa['Date'], format='%d-%b-%Y %H:%M')
gapa['day'] = gapa['time'].dt.date
gapa['dayhour'] = gapa['time'].dt.strftime('%Y-%m-%d %H')
gapa.rename(columns={'PMHR10':'rpm10'}, inplace=True)

# Sort chronologically
gapa = gapa.sort_values('time').reset_index(drop=True).dropna(subset=['rpm10'])


# last 3/4
# spl_index = int(len(gapa) * 0.25)
# gapa = gapa[spl_index:]

h_gapa = gapa.groupby('dayhour').agg(
    rhourlyPM10=('rpm10', lambda x: x.mean(skipna=True)),
).reset_index()

d_gapa = gapa.groupby('day').agg(
    rdailyPM10=('rpm10', lambda x: pd.to_numeric(x, errors='coerce').mean(skipna=True)),
).reset_index()




In [ ]:
hmodel_00589 = train_hourly_model(quantAQ00589, h_gapa)
dmodel_00589 = train_daily_model(quantAQ00589, d_gapa)
hmodel_00590 = train_hourly_model(quantAQ00590, h_gapa)
dmodel_00590 = train_daily_model(quantAQ00590, d_gapa)
hmodel_00591 = train_hourly_model(quantAQ00591, h_gapa)
dmodel_00591 = train_daily_model(quantAQ00591, d_gapa)
hmodel_00592 = train_hourly_model(quantAQ00592, h_gapa)
dmodel_00592 = train_daily_model(quantAQ00592, d_gapa)
hmodel_00593 = train_hourly_model(quantAQ00593, h_gapa)
dmodel_00593 = train_daily_model(quantAQ00593, d_gapa)
hmodel_00595 = train_hourly_model(quantAQ00595, h_gapa)
dmodel_00595 = train_daily_model(quantAQ00595, d_gapa)

In [ ]:
# save models
joblib.dump(hmodel_00589, 'models/hmodel_00589.joblib')
joblib.dump(dmodel_00589, 'models/dmodel_00589.joblib')
joblib.dump(hmodel_00590, 'models/hmodel_00590.joblib')
joblib.dump(dmodel_00590, 'models/dmodel_00590.joblib')
joblib.dump(hmodel_00591, 'models/hmodel_00591.joblib')
joblib.dump(dmodel_00591, 'models/dmodel_00591.joblib')
joblib.dump(hmodel_00592, 'models/hmodel_00592.joblib')
joblib.dump(dmodel_00592, 'models/dmodel_00592.joblib')
joblib.dump(hmodel_00593, 'models/hmodel_00593.joblib')
joblib.dump(dmodel_00593, 'models/dmodel_00593.joblib')
joblib.dump(hmodel_00595, 'models/hmodel_00595.joblib')
joblib.dump(dmodel_00595, 'models/dmodel_00595.joblib')
